# Modelo

Para ver como cambia la performance del modelo con el tratamiento de los datos primero hay que tener un modelo de partida que optimizar.

In [95]:
import librosa
import numpy as np
import pandas as pd
import os
import joblib


from scipy.signal import butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, classification_report, confusion_matrix, roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.decomposition import PCA

from sklearn.base import BaseEstimator, TransformerMixin

In [96]:
CUTOFF_HZ = 4_000
SAMPLING_RATE = 16_000

In [97]:
train = pd.read_csv('../dataset/filtrado/train.csv')
test = pd.read_csv('../dataset/filtrado/test.csv')

In [98]:
train.head()

,cycle_wav_file,label
0,158_1p3_Pr_mc_AKGC417L_cycle4.wav,1
1,146_8p3_Lr_mc_AKGC417L_cycle4.wav,1
2,154_1b3_Tc_mc_AKGC417L_cycle4.wav,0
3,158_1p4_Pr_mc_AKGC417L_cycle2.wav,1
4,207_3b2_Pl_mc_AKGC417L_cycle6.wav,0


## Preprocesamiento

In [99]:
class AudioFilterResample(BaseEstimator, TransformerMixin):
    def __init__(self, sr_target=SAMPLING_RATE, highcut=CUTOFF_HZ, lowcut=0, order=6):
        self.sr_target = sr_target
        self.highcut = highcut
        self.lowcut = lowcut
        self.order = order

    def _filter(self, y, sr):
        nyq = 0.5 * sr
        if self.lowcut == 0:
            normal_cut = self.highcut / nyq
            b, a = butter(self.order, normal_cut, btype='low')
        else:
            normal_low = self.lowcut / nyq
            normal_high = self.highcut / nyq
            b, a = butter(self.order, [normal_low, normal_high], btype='band')
        return filtfilt(b, a, y)

    def _filter_and_resample(self, path):
        y, sr = librosa.load(path, sr=None)
        y = self._filter(y, sr)
        if sr != self.sr_target:
            y = librosa.resample(y, orig_sr=sr, target_sr=self.sr_target)
        return y, self.sr_target

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        processed = []
        for path in X:
            y, sr = self._filter_and_resample(path)
            processed.append(y)
        return processed

## Espectrogramas

Se suelen usar Mel espectrogramas

In [100]:
class MelSpectrogramTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, sr=SAMPLING_RATE, n_mels=128):
        self.sr = sr
        self.n_mels = n_mels

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        specs = []
        for y in X:
            S = librosa.feature.melspectrogram(y=y, sr=self.sr, n_mels=self.n_mels)
            S_dB = librosa.power_to_db(S, ref=np.max)
            specs.append(S_dB)
        return specs
    
class PadSpectrograms(BaseEstimator, TransformerMixin):
    def __init__(self, max_length=245): # TODO: ajustar max_length
        self.max_length = max_length

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        padded = []
        for mel in X:
            pad_width = self.max_length - mel.shape[1]
            if pad_width > 0:
                mel_padded = np.pad(mel, ((0, 0), (0, pad_width)), mode='constant')
            else:
                mel_padded = mel[:, :self.max_length]
            padded.append(mel_padded)
        return np.array(padded)
    
class FlattenTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        n_samples = X.shape[0]
        return X.reshape(n_samples, -1)

### Modelos clásicos

#### Random Forest

##### Entrenamiento

In [102]:
pipeline_preproc = Pipeline([
    ('filter_resample', AudioFilterResample()),
    ('melspec', MelSpectrogramTransformer()),
    ('pad', PadSpectrograms(max_length=245)),
    ('flatten', FlattenTransformer()),
    ('scaler', StandardScaler())
])

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid = {
    'max_depth': [6, 7, 8],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [10, 12, 15]
}

In [103]:
X_train_paths = ['../dataset/ciclos/' + f for f in train['cycle_wav_file']]
X_test_paths = ['../dataset/ciclos/' + f for f in test['cycle_wav_file']]
y_train = train['label'].values
y_test = test['label'].values

In [104]:
X_train_proc = pipeline_preproc.fit_transform(X_train_paths)
X_test_proc = pipeline_preproc.transform(X_test_paths)

In [105]:
grid_search = GridSearchCV(rf, param_grid, cv=StratifiedKFold(n_splits=5), scoring='roc_auc', n_jobs=1, verbose=1) # o auc-roc o recall?
grid_search.fit(X_train_proc, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [6, 7, ...], 'min_samples_leaf': [10, 12, ...], 'min_samples_split': [10, 15, ...]}"
,scoring,'roc_auc'
,n_jobs,1
,refit,True
,cv,StratifiedKFo...shuffle=False)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [107]:
pd.DataFrame(grid_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
20,8,10,20,0.750301
19,8,10,15,0.750301
18,8,10,10,0.750301
21,8,12,10,0.748967
22,8,12,15,0.748967
23,8,12,20,0.748967
24,8,15,10,0.747925
25,8,15,15,0.747925
26,8,15,20,0.747925
13,7,12,15,0.745859


In [108]:
print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

best_model = grid_search.best_estimator_

Best params: {'max_depth': 8, 'min_samples_leaf': 10, 'min_samples_split': 10}
Best CV score: 0.7503010629279294


##### Evaluación

In [109]:
y_pred = best_model.predict(X_test_proc)
print(classification_report(y_test, y_pred))
print('AUC-ROC: ', round(roc_auc_score(y_test, y_pred), 2))
print('Recall: ', round(recall_score(y_test, y_pred), 2))

              precision    recall  f1-score   support

           0       0.71      0.52      0.60       429
           1       0.66      0.82      0.73       493

    accuracy                           0.68       922
   macro avg       0.69      0.67      0.67       922
weighted avg       0.69      0.68      0.67       922

AUC-ROC:  0.67
Recall:  0.82


In [110]:
pipeline_final = Pipeline([
    ('preproc', pipeline_preproc),
    ('rf', best_model)
])

pipeline_final.fit(X_train_paths, y_train)

,steps,"[('preproc', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('filter_resample', ...), ('melspec', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,sr_target,16000
,highcut,4000
,lowcut,0


In [111]:
joblib.dump(pipeline_final, '../modelos/melspec_filt_rf.pkl')

['../modelos/melspec_filt_rf.pkl']

##### Evaluación en audios propios

In [112]:
model = joblib.load('./melspec_filt_rf.pkl')

In [113]:
path = '../audios_propios/ciclos_energy_based'
files = os.listdir(path)

df_audios_propios = pd.DataFrame(files, columns=['cycle_wav_file'])

In [114]:
df_audios_propios['label'] = np.where(df_audios_propios['cycle_wav_file'].str.contains('respiracion4_ciclo_5'), 1, 0)

In [115]:
df_audios_propios # audios de validacion

,cycle_wav_file,label
0,respiracion1_ciclo_1.wav,0
1,respiracion1_ciclo_2.wav,0
2,respiracion1_ciclo_3.wav,0
3,respiracion1_ciclo_4.wav,0
4,respiracion1_ciclo_5.wav,0
5,respiracion1_ciclo_6.wav,0
6,respiracion1_ciclo_7.wav,0
7,respiracion2_ciclo_1.wav,0
8,respiracion2_ciclo_2.wav,0
9,respiracion2_ciclo_3.wav,0


cualquier ciclo del audio 4 puede ser positivo

In [116]:
archivos = df_audios_propios['cycle_wav_file'].tolist()

X_propios_paths = [os.path.join(path, f) for f in archivos]
y_propios = df_audios_propios['label'].values

In [117]:
y_propios_pred = model.predict(X_propios_paths)

In [118]:
y_propios_proba_pred = model.predict_proba(X_propios_paths)[:, 1]

In [119]:
print(classification_report(y_propios, y_propios_pred))

              precision    recall  f1-score   support

           0       1.00      0.96      0.98        48
           1       0.33      1.00      0.50         1

    accuracy                           0.96        49
   macro avg       0.67      0.98      0.74        49
weighted avg       0.99      0.96      0.97        49



In [120]:
df_audios_propios['predicted'] = y_propios_pred
df_audios_propios['probability'] = y_propios_proba_pred.round(2)

In [90]:
df_audios_propios

,cycle_wav_file,label,predicted,probability
0,respiracion1_ciclo_1.wav,0,0,0.46
1,respiracion1_ciclo_2.wav,0,0,0.12
2,respiracion1_ciclo_3.wav,0,0,0.34
3,respiracion1_ciclo_4.wav,0,0,0.08
4,respiracion1_ciclo_5.wav,0,0,0.28
5,respiracion1_ciclo_6.wav,0,0,0.30
6,respiracion1_ciclo_7.wav,0,0,0.31
7,respiracion2_ciclo_1.wav,0,0,0.14
8,respiracion2_ciclo_2.wav,0,0,0.36
9,respiracion2_ciclo_3.wav,0,0,0.37


Parece prometedor, los ciclos del audio 4 fueron clasificados como anormales.

### SVM

In [ ]:
from sklearn.svm import SVC